# Num. Epoch = 5

## P(Y|X) - scaling 0

In [3]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []
profile_dists  = []
reference_dists = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))
    profile_dists.append(profile_dist)
    reference_dists.append(ref_dist)

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

# ----------------------------------------------------------------------

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 54502.56it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.060076
mean ε  : 0.525845
median ε: 0.516835
std ε   : 0.105744
min ε   : 0.194137


In [25]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15305.54it/s]



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.477128
mean ε  : 0.288278
median ε: 0.302942
std ε   : 0.073890
min ε   : 0.030101


## P(Y|X) - scaling 1

In [12]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39996.04it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.028281
mean ε  : 0.479541
median ε: 0.466843
std ε   : 0.113312
min ε   : 0.159313


In [26]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15727.56it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.517277
mean ε  : 0.331835
median ε: 0.355768
std ε   : 0.083875
min ε   : 0.009908


## P(Y|X) - scaling 2

In [13]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 56658.72it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.016167
mean ε  : 0.458475
median ε: 0.447017
std ε   : 0.108352
min ε   : 0.136353


In [27]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:08<00:00, 5239.90it/s] 


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.534622
mean ε  : 0.352070
median ε: 0.368177
std ε   : 0.077561
min ε   : 0.006691
